# 1 · Fine-tuned encoders with two tagging heads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emrtunali/yetkin/blob/main/notebooks/01_encoder.ipynb)

Two pre-trained encoders, each fine-tuned with **one BIO head per span type** (skill, knowledge) on a shared
body, three seeds each. JobBERT was further pre-trained on job postings (Zhang et al., 2022); DeBERTa-v3 is a
strong general encoder. The epoch with the best dev F1 is kept; test predictions go to Google Drive.

About 25 minutes on a T4.

In [ ]:
# Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
!pip -q install "yetkin[train,qlora,track] @ git+https://github.com/emrtunali/yetkin.git"

import json, os, time
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/yetkin"          # results survive a disconnect here
os.makedirs(f"{OUT}/predictions", exist_ok=True)

def secret(name):
    """Optional keys from the 🔑 Secrets panel: HF_TOKEN (to publish models), WANDB_API_KEY (to log runs)."""
    try:
        return userdata.get(name)
    except Exception:
        return None

In [ ]:
# Experiment tracking: Weights & Biases when a WANDB_API_KEY secret exists, otherwise just printed.
wandb = None
if secret("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
    import wandb
print("W&B logging:", "on" if wandb else "off")

In [ ]:
from yetkin.data import load, chunks
train = [c for s in load("train") for c in chunks(s, 128)]
dev, test = load("dev"), load("test")
print(f"train {len(train):,} sentences · dev {len(dev):,} · test {len(test):,}")

In [ ]:
MODELS = {"JobBERT": ("jjzha/jobbert-base-cased", 3e-5), "DeBERTa-v3-base": ("microsoft/deberta-v3-base", 2e-5)}
SEEDS = [1, 2, 3]
EPOCHS, BATCH = 5, 16

In [ ]:
from yetkin import encoder
from yetkin.io import save_predictions
from yetkin.metrics import score

runs = []
for label, (base, lr) in MODELS.items():
    for seed in SEEDS:
        slug = f"{label.lower().replace(' ', '-')}-seed{seed}"
        path = f"{OUT}/predictions/{slug}.jsonl"
        if os.path.exists(path):
            print("done already:", slug); continue
        run = wandb.init(project="yetkin", name=slug, config={"base": base, "lr": lr, "seed": seed, "epochs": EPOCHS}) if wandb else None
        t0 = time.time()
        model, tok, info = encoder.train(base, train, dev, epochs=EPOCHS, batch_size=BATCH, lr=lr, seed=seed,
                                         on_epoch=(lambda h: run.log({"loss": h["loss"], "dev_f1": h["dev_f1"]})) if run else None)
        minutes = (time.time() - t0) / 60
        t1 = time.time()
        pred = encoder.predict(model, tok, test)
        per_1k = (time.time() - t1) / len(test) * 1000
        s = score([x.spans for x in test], pred)
        meta = {"name": f"{label} · seed {seed}", "system": label, "seed": seed, "base": base, "lr": lr, "epochs": EPOCHS,
                "best_dev_f1": round(info["best_dev_f1"], 4), "train_minutes": round(minutes, 1), "seconds_per_1k_sentences": round(per_1k, 2)}
        save_predictions(path, test, pred, meta=meta)
        model.save(f"/content/runs/{slug}", tok)            # local disk; the best one is copied to Drive below
        runs.append({**meta, "test_f1": s["strict_micro"]["f1"]})
        print(f"{slug}: dev {info['best_dev_f1']:.3f}  test {s['strict_micro']['f1']:.3f}  ({minutes:.1f} min)")
        if run: run.log({"test_f1": s["strict_micro"]["f1"]}); run.finish()
        del model; torch.cuda.empty_cache()

In [ ]:
# Keep the run with the best *dev* score (choosing on test would be cheating) and publish it if you like.
import shutil
if runs:
    best = max(runs, key=lambda r: r["best_dev_f1"])
    slug = f"{best['system'].lower().replace(' ', '-')}-seed{best['seed']}"
    shutil.copytree(f"/content/runs/{slug}", f"{OUT}/models/encoder", dirs_exist_ok=True)
    json.dump(best, open(f"{OUT}/models/encoder/run.json", "w"), indent=1)
    print("best on dev:", best["name"], f"(test {best['test_f1']:.3f})")
    if secret("HF_TOKEN"):
        from huggingface_hub import HfApi
        api = HfApi(token=secret("HF_TOKEN"))
        repo = api.create_repo("yetkin-skill-tagger", exist_ok=True).repo_id
        api.upload_folder(folder_path=f"{OUT}/models/encoder", repo_id=repo)
        print("published:", f"https://huggingface.co/{repo}")